# 面试问题：Elastic 分布式训练怎样在 world size 改变后恢复模型、优化器与数据进度？

        ## 可直接复述的回答主线

        1. Elastic checkpoint 不能只保存 rank-local 文件名，而要保存每个全局张量的 shape、dtype、分片区间和校验和。
2. 恢复时先验证全部 shard，再按全局坐标重建逻辑张量，最后根据新的 world size 重新分片。
3. 除模型参数外还要恢复优化器动量、global step、sampler cursor 和随机数状态，否则训练轨迹会静默分叉。
4. 直接让新 rank 读取同编号旧 rank shard 会在 world size 改变时丢参数或重复参数。
5. 正确性应通过恢复张量差、同一下一批次的 forward/backward 结果和逐样本预测共同验证。
6. 生产还需要原子 manifest、对象存储一致性、两阶段提交、冗余、异步保存、拓扑映射和故障演练。

        后续实验会在同一批输入上依次展示朴素基线、手写核心机制、中间过程、失败修正和生产边界。

## 1. 真实案例与输入预览

案例使用六条脱敏的四维训练样本和一个手写 PyTorch 线性回归器。模型在 world size=3 的教学作业中训练三步后分片保存，随后模拟一个 worker 故障并以 world size=2 恢复，再与不中断参考轨迹执行同一下一批次。

In [1]:
import hashlib  # 为每个分片计算真实 SHA-256 内容校验和。
import math  # 汇总实际 backward 产生的梯度范数。
import numpy as np  # 手写全局张量分片、重组和重分片。
import torch  # 执行真实模型 forward、backward 和动量更新。
torch.manual_seed(58)  # 固定模型初始化和训练轨迹。
samples = [{"id": "train-01", "features": [1.0, 0.2, -0.1, 0.5], "target": 1.82}, {"id": "train-02", "features": [0.3, 1.1, 0.4, -0.2], "target": -0.14}, {"id": "train-03", "features": [-0.5, 0.7, 1.2, 0.1], "target": 0.13}, {"id": "train-04", "features": [0.8, -0.4, 0.3, 1.0], "target": 2.29}, {"id": "train-05", "features": [0.1, 0.5, -0.7, 1.3], "target": 0.88}, {"id": "train-06", "features": [1.2, -0.2, 0.6, -0.5], "target": 2.16}]  # 定义六条具有可读 ID、特征和回归目标的训练样本。
features = torch.tensor([sample["features"] for sample in samples], dtype=torch.float64)  # 构造六乘四训练特征张量。
targets = torch.tensor([sample["target"] for sample in samples], dtype=torch.float64)  # 构造六维回归目标。
class TinyRegressor(torch.nn.Module):  # 定义具有显式参数和 forward 的教学训练模型。
    def __init__(self):  # 初始化四维权重和标量偏置。
        super().__init__()  # 注册 PyTorch 参数管理。
        self.weight = torch.nn.Parameter(torch.randn(4, dtype=torch.float64) * 0.2)  # 创建四维可训练权重。
        self.bias = torch.nn.Parameter(torch.zeros(1, dtype=torch.float64))  # 创建可训练标量偏置。
    def forward(self, inputs):  # 对一批四维样本执行线性前向。
        return inputs @ self.weight + self.bias  # 返回每条样本的预测值。
def fresh_momentum(model):  # 为模型参数创建与形状一致的动量状态。
    return {name: torch.zeros_like(parameter) for name, parameter in model.named_parameters()}  # 返回 weight 和 bias 的零动量。
def train_step(model, momentum, batch_indices, learning_rate=0.08, momentum_rate=0.7):  # 手写一次带动量的真实训练更新。
    model.zero_grad(set_to_none=True)  # 清除上一步参数梯度。
    predictions = model(features[batch_indices])  # 调用显式 forward 计算当前小批次预测。
    loss = ((predictions - targets[batch_indices]) ** 2).mean()  # 计算小批次均方误差。
    loss.backward()  # 对 weight 和 bias 执行真实反向传播。
    gradient_norm = math.sqrt(sum(float((parameter.grad ** 2).sum().item()) for parameter in model.parameters()))  # 汇总本步梯度二范数。
    with torch.no_grad():  # 在无梯度上下文中更新动量和参数。
        for name, parameter in model.named_parameters():  # 遍历两个模型参数。
            momentum[name].mul_(momentum_rate).add_(parameter.grad)  # 更新需要一起 checkpoint 的优化器动量。
            parameter.add_(momentum[name], alpha=-learning_rate)  # 用新动量执行参数更新。
    return loss.item(), gradient_norm  # 返回可观测损失和梯度范数。
model_before_failure = TinyRegressor()  # 创建模拟分布式作业的初始模型。
momentum_before_failure = fresh_momentum(model_before_failure)  # 创建对应优化器动量。
batches = [torch.tensor([0, 1]), torch.tensor([2, 3]), torch.tensor([4, 5]), torch.tensor([0, 2])]  # 定义前三个保存前批次和一个恢复后批次。
pre_checkpoint_history = []  # 保存故障前真实训练轨迹。
for step, batch_indices in enumerate(batches[:3], start=1):  # 在 checkpoint 前执行三次 forward/backward。
    loss, gradient_norm = train_step(model_before_failure, momentum_before_failure, batch_indices)  # 训练当前小批次并更新动量。
    pre_checkpoint_history.append({"step": step, "batch": batch_indices.tolist(), "loss": loss, "gradient_norm": gradient_norm})  # 保存本步训练证据。
checkpoint_state = {"model.weight": model_before_failure.weight.detach().cpu().numpy().copy(), "model.bias": model_before_failure.bias.detach().cpu().numpy().copy(), "momentum.weight": momentum_before_failure["weight"].detach().cpu().numpy().copy(), "momentum.bias": momentum_before_failure["bias"].detach().cpu().numpy().copy()}  # 捕获模型和优化器的全局逻辑张量。
checkpoint_metadata = {"global_step": 3, "sampler_cursor": 6, "old_world_size": 3, "rng_seed": 58}  # 保存恢复训练顺序所需的作业进度元数据。
print("教学实验输入：六条训练样本与故障前轨迹")  # 标记下方为小型真实梯度实验。
for sample in samples:  # 逐条展示训练输入和目标。
    print(f"{sample['id']} features={sample['features']} target={sample['target']}")  # 输出当前回归样本。
print("故障前训练轨迹：", pre_checkpoint_history)  # 展示前三次 forward/backward 的损失和梯度。
print("checkpoint元数据：", checkpoint_metadata, "张量形状=", {name: value.shape for name, value in checkpoint_state.items()})  # 展示保存范围和全局 shape。

教学实验输入：六条训练样本与故障前轨迹
train-01 features=[1.0, 0.2, -0.1, 0.5] target=1.82
train-02 features=[0.3, 1.1, 0.4, -0.2] target=-0.14
train-03 features=[-0.5, 0.7, 1.2, 0.1] target=0.13
train-04 features=[0.8, -0.4, 0.3, 1.0] target=2.29
train-05 features=[0.1, 0.5, -0.7, 1.3] target=0.88
train-06 features=[1.2, -0.2, 0.6, -0.5] target=2.16
故障前训练轨迹： [{'step': 1, 'batch': [0, 1], 'loss': 1.9557595699679895, 'gradient_norm': 2.779426063846315}, {'step': 2, 'batch': [2, 3], 'loss': 2.0306901623804228, 'gradient_norm': 3.292246628474718}, {'step': 3, 'batch': [4, 5], 'loss': 0.8496553569116991, 'gradient_norm': 2.3051844906429544}]
checkpoint元数据： {'global_step': 3, 'sampler_cursor': 6, 'old_world_size': 3, 'rng_seed': 58} 张量形状= {'model.weight': (4,), 'model.bias': (1,), 'momentum.weight': (4,), 'momentum.bias': (1,)}


## 2. Baseline / 基线：新 rank 直接读取同编号旧 rank shard

旧 world size=3 时四维 weight 被分成 `[2,1,1]`。新 world size=2 只读取旧 rank0、rank1，最后一个参数被遗漏后补零；文件都存在，但模型已静默损坏。

In [2]:
def old_rank_local_shards(state, world_size):  # 按旧 rank 编号生成没有全局坐标的朴素分片。
    rank_files = {rank: {} for rank in range(world_size)}  # 为每个旧 rank 创建本地 checkpoint 字典。
    for name, array in state.items():  # 逐全局模型和动量张量切分。
        pieces = np.array_split(array.reshape(-1), world_size)  # 按旧 world size 连续切分扁平数组。
        for rank, piece in enumerate(pieces):  # 把每段写到对应 rank 文件。
            rank_files[rank][name] = piece.copy()  # 保存仅含值但没有 start/end 的本地 shard。
    return rank_files  # 返回旧拓扑的 rank-local 文件集合。
old_rank_files = old_rank_local_shards(checkpoint_state, checkpoint_metadata["old_world_size"])  # 生成 world size=3 的朴素 checkpoint。
def naive_same_rank_recovery(rank_files, template_state, new_world_size):  # 模拟新 rank 只读同编号旧 shard 的错误恢复。
    recovered = {}  # 保存错误重建的全局张量。
    for name, template in template_state.items():  # 逐张量拼接新 rank 能看到的旧文件。
        visible_parts = [rank_files[rank][name] for rank in range(new_world_size)]  # 只读取旧 rank0 和 rank1，遗漏旧 rank2。
        visible = np.concatenate(visible_parts) if visible_parts else np.array([], dtype=template.dtype)  # 拼接可见旧分片。
        padded = np.zeros(template.size, dtype=template.dtype)  # 创建目标大小并用零掩盖缺失数据。
        padded[: min(template.size, visible.size)] = visible[: template.size]  # 把错误可见片段复制到全局张量前部。
        recovered[name] = padded.reshape(template.shape)  # 恢复原 shape 但无法恢复遗漏值。
    return recovered  # 返回表面 shape 正确的损坏状态。
baseline_recovered_state = naive_same_rank_recovery(old_rank_files, checkpoint_state, new_world_size=2)  # 用错误 rank 映射恢复到两个 worker。
baseline_parameter_error = max(float(np.max(np.abs(baseline_recovered_state[name] - checkpoint_state[name]))) for name in checkpoint_state)  # 计算所有模型和动量张量的最大误差。
print("Baseline 同编号 rank 恢复")  # 标记下表展示静默遗漏。
print("tensor             expected                         recovered                        max_error")  # 输出逐张量恢复对照表头。
for name in checkpoint_state:  # 逐张量展示原值与错误恢复值。
    tensor_error = float(np.max(np.abs(baseline_recovered_state[name] - checkpoint_state[name])))  # 计算当前张量最大误差。
    print(f"{name:<18} {np.round(checkpoint_state[name], 5).tolist()!s:<32} {np.round(baseline_recovered_state[name], 5).tolist()!s:<32} {tensor_error:.6f}")  # 输出当前张量的丢失内容。

Baseline 同编号 rank 恢复
tensor             expected                         recovered                        max_error
model.weight       [0.67719, -0.18291, 0.37224, 0.29614] [0.67719, -0.18291, 0.37224, 0.0] 0.296144
model.bias         [0.67446]                        [0.67446]                        0.000000
momentum.weight    [-3.58639, 0.5591, -0.28137, -2.10937] [-3.58639, 0.5591, -0.28137, 0.0] 2.109374
momentum.bias      [-3.7993]                        [-3.7993]                        0.000000


## 3. 底层实现：全局坐标 Manifest、SHA-256 验证与重新分片

每个 shard 保存 tensor、start、end、shape、dtype、rank 和内容摘要。恢复器先验证摘要与区间连续性，重建全局逻辑张量后再按新 world size=2 切分。

In [3]:
def array_checksum(array):  # 对连续数组原始字节计算 SHA-256。
    contiguous = np.ascontiguousarray(array)  # 固定内存布局以保证摘要稳定。
    return hashlib.sha256(contiguous.tobytes()).hexdigest()  # 返回六十四位内容摘要。
def create_manifest(state, world_size, metadata):  # 为全局逻辑张量创建带坐标的分布式 checkpoint。
    manifest = {"format_version": 1, "world_size": world_size, "global_step": metadata["global_step"], "sampler_cursor": metadata["sampler_cursor"], "tensors": {}}  # 初始化全局作业和张量清单。
    shard_store = {}  # 模拟对象存储中的独立 shard 内容。
    for name, array in state.items():  # 逐全局张量生成连续坐标分片。
        flat = array.reshape(-1)  # 把任意 shape 转为一维逻辑坐标。
        pieces = np.array_split(flat, world_size)  # 按旧 world size 生成大小近似的连续分片。
        entries = []  # 保存当前张量的分片清单。
        start = 0  # 初始化当前张量的全局起点。
        for rank, piece in enumerate(pieces):  # 逐旧 rank 记录分片值和坐标。
            end = start + piece.size  # 计算当前分片右开区间终点。
            shard_id = f"{name}/rank-{rank}"  # 构造稳定的对象存储键。
            shard_store[shard_id] = piece.copy()  # 保存实际分片字节内容。
            entries.append({"shard_id": shard_id, "rank": rank, "start": start, "end": end, "sha256": array_checksum(piece)})  # 保存坐标、所有者和真实摘要。
            start = end  # 推进下一分片全局起点。
        manifest["tensors"][name] = {"shape": list(array.shape), "dtype": str(array.dtype), "numel": int(array.size), "shards": entries}  # 保存当前全局张量元数据。
    return manifest, shard_store  # 返回原子清单内容和分片对象。
def verify_and_reconstruct(manifest, shard_store):  # 验证全部 shard 并重建全局逻辑张量。
    reconstructed = {}  # 保存验证后的全局状态。
    for name, tensor_meta in manifest["tensors"].items():  # 逐张量检查坐标和内容。
        ordered_entries = sorted(tensor_meta["shards"], key=lambda entry: entry["start"])  # 按全局起点排列分片。
        expected_start = 0  # 要求第一个分片从零开始且后续连续。
        pieces = []  # 保存通过验证的分片值。
        for entry in ordered_entries:  # 逐分片执行存在性、摘要和区间验证。
            if entry["shard_id"] not in shard_store:  # 检查对象是否实际存在。
                raise ValueError(f"missing_shard:{entry['shard_id']}")  # 对缺失对象快速失败。
            piece = shard_store[entry["shard_id"]]  # 读取当前分片内容。
            if array_checksum(piece) != entry["sha256"]:  # 重新计算 SHA-256 而不是只看文件存在。
                raise ValueError(f"checksum_mismatch:{entry['shard_id']}")  # 阻止损坏数据进入模型。
            if entry["start"] != expected_start or entry["end"] - entry["start"] != piece.size:  # 检查区间连续且长度与内容一致。
                raise ValueError(f"range_mismatch:{entry['shard_id']}")  # 阻止重复或缺口坐标。
            pieces.append(piece)  # 保存验证通过的分片。
            expected_start = entry["end"]  # 推进期望的下一全局坐标。
        if expected_start != tensor_meta["numel"]:  # 检查分片是否覆盖完整全局张量。
            raise ValueError(f"incomplete_tensor:{name}")  # 拒绝尾部缺失的清单。
        flat = np.concatenate(pieces).astype(np.dtype(tensor_meta["dtype"]), copy=False)  # 拼接并恢复声明 dtype。
        reconstructed[name] = flat.reshape(tensor_meta["shape"])  # 恢复声明的全局 shape。
    return reconstructed  # 返回模型和优化器的完整全局状态。
manifest, shard_store = create_manifest(checkpoint_state, checkpoint_metadata["old_world_size"], checkpoint_metadata)  # 创建 world size=3 的可验证 checkpoint。
corrected_recovered_state = verify_and_reconstruct(manifest, shard_store)  # 验证并恢复完整逻辑张量。
new_world_shards = {rank: {} for rank in range(2)}  # 初始化 world size=2 的新拓扑分片。
for name, array in corrected_recovered_state.items():  # 逐全局张量按新 worker 数重新切分。
    for rank, piece in enumerate(np.array_split(array.reshape(-1), 2)):  # 生成两个连续新 shard。
        new_world_shards[rank][name] = piece.copy()  # 保存新 rank 的参数或动量片段。
print("weight Manifest 分片中间量")  # 标记下表展示真实坐标和摘要。
for entry in manifest["tensors"]["model.weight"]["shards"]:  # 逐旧 rank 展示 weight 分片。
    print({**entry, "sha256": entry["sha256"][:12], "values": np.round(shard_store[entry["shard_id"]], 6).tolist()})  # 输出坐标、摘要前缀和真实值。
print("恢复后按world_size=2重分片：", {rank: {name: value.tolist() for name, value in tensors.items()} for rank, tensors in new_world_shards.items()})  # 展示新拓扑的实际 shard。

weight Manifest 分片中间量
{'shard_id': 'model.weight/rank-0', 'rank': 0, 'start': 0, 'end': 2, 'sha256': 'a0e369d44278', 'values': [0.67719, -0.182913]}
{'shard_id': 'model.weight/rank-1', 'rank': 1, 'start': 2, 'end': 3, 'sha256': 'c94ce31d90c3', 'values': [0.372245]}
{'shard_id': 'model.weight/rank-2', 'rank': 2, 'start': 3, 'end': 4, 'sha256': '9488204f9e01', 'values': [0.296144]}
恢复后按world_size=2重分片： {0: {'model.weight': [0.6771903376784065, -0.18291277671481687], 'model.bias': [0.674458375139786], 'momentum.weight': [-3.586390517931875, 0.5590967543342278], 'momentum.bias': [-3.7992958012608296]}, 1: {'model.weight': [0.3722448228245603, 0.29614369714107097], 'model.bias': [], 'momentum.weight': [-0.28136958375222804, -2.1093739283290374], 'momentum.bias': []}}


## 4. 恢复后继续训练与结果解读

将不中断参考、朴素恢复和 Manifest 恢复都加载到同一模型，并对相同下一批次 `[train-01, train-03]` 做一次真实 forward/backward。逐样本预测能直接看到静默状态损坏的后果。

In [4]:
def load_training_state(state):  # 把 NumPy checkpoint 恢复为 PyTorch 模型和动量。
    model = TinyRegressor()  # 创建目标模型实例。
    with torch.no_grad():  # 在无梯度环境复制 checkpoint 参数。
        model.weight.copy_(torch.from_numpy(state["model.weight"]))  # 恢复四维模型权重。
        model.bias.copy_(torch.from_numpy(state["model.bias"]))  # 恢复模型偏置。
    momentum = {"weight": torch.from_numpy(state["momentum.weight"].copy()), "bias": torch.from_numpy(state["momentum.bias"].copy())}  # 恢复与参数同形状的优化器动量。
    return model, momentum  # 返回可继续训练的完整状态。
reference_model, reference_momentum = load_training_state(checkpoint_state)  # 构造不中断参考分支。
baseline_model, baseline_momentum = load_training_state(baseline_recovered_state)  # 构造同编号 rank 错误恢复分支。
corrected_model, corrected_momentum = load_training_state(corrected_recovered_state)  # 构造 Manifest 正确恢复分支。
continuation_batch = batches[3]  # 读取 checkpoint 元数据指向的下一训练批次。
reference_loss, reference_gradient_norm = train_step(reference_model, reference_momentum, continuation_batch)  # 对参考分支执行下一步真实训练。
baseline_loss, baseline_gradient_norm = train_step(baseline_model, baseline_momentum, continuation_batch)  # 对错误恢复分支执行相同训练。
corrected_loss, corrected_gradient_norm = train_step(corrected_model, corrected_momentum, continuation_batch)  # 对正确恢复分支执行相同训练。
with torch.no_grad():  # 在恢复后一步评估全部六条样本。
    reference_predictions = reference_model(features)  # 计算不中断参考预测。
    baseline_predictions = baseline_model(features)  # 计算错误恢复预测。
    corrected_predictions = corrected_model(features)  # 计算 Manifest 恢复预测。
baseline_resume_gap = float(torch.max(torch.abs(baseline_predictions - reference_predictions)).item())  # 计算错误恢复相对参考的最大预测偏差。
corrected_resume_gap = float(torch.max(torch.abs(corrected_predictions - reference_predictions)).item())  # 计算正确恢复相对参考的最大预测偏差。
print("样本      target   reference   same-rank baseline   manifest recovery")  # 输出逐样本恢复后预测表头。
for index, sample in enumerate(samples):  # 逐训练样本展示三条恢复轨迹。
    print(f"{sample['id']:<9} {sample['target']:>7.3f} {reference_predictions[index].item():>11.5f} {baseline_predictions[index].item():>20.5f} {corrected_predictions[index].item():>19.5f}")  # 输出当前样本的真实预测差异。
print(f"结果解读：下一步loss参考={reference_loss:.6f}、错误恢复={baseline_loss:.6f}、Manifest恢复={corrected_loss:.6f}；预测最大偏差分别为{baseline_resume_gap:.6f}和{corrected_resume_gap:.6f}。")  # 解释完整状态恢复对训练连续性的影响。

样本      target   reference   same-rank baseline   manifest recovery
train-01    1.820     1.93798              1.75944             1.93798
train-02   -0.140     0.93863              1.04276             0.93863
train-03    0.130     0.68183              0.65472             0.68183
train-04    2.290     2.24047              1.85399             2.24047
train-05    0.880     1.16924              0.65483             1.16924
train-06    2.160     2.02646              2.25771             2.02646
结果解读：下一步loss参考=0.231183、错误恢复=0.284527、Manifest恢复=0.231183；预测最大偏差分别为0.514408和0.000000。


## 5. 失败案例与修正：对象存在但 shard 内容已损坏

把一个 weight shard 的首元素改写后，存在性检查仍通过；带 SHA-256 的恢复器会在加载模型前报 `checksum_mismatch`，随后使用原始冗余副本恢复。

In [5]:
corrupted_store = {shard_id: value.copy() for shard_id, value in shard_store.items()}  # 深拷贝分片对象以模拟存储损坏。
corrupted_shard_id = manifest["tensors"]["model.weight"]["shards"][0]["shard_id"]  # 选择第一个模型 weight shard。
corrupted_store[corrupted_shard_id][0] += 0.5  # 修改一个真实参数值但保留旧摘要。
existence_only_accepts = all(entry["shard_id"] in corrupted_store for tensor_meta in manifest["tensors"].values() for entry in tensor_meta["shards"])  # 模拟只检查对象存在性的错误门禁。
corruption_detected = False  # 初始化真实校验检测标志。
corruption_reason = ""  # 保存恢复器给出的具体失败原因。
try:  # 尝试从损坏对象重建全局状态。
    verify_and_reconstruct(manifest, corrupted_store)  # 重新计算所有 shard 摘要。
except ValueError as error:  # 捕获预期的校验失败而不把异常保存到 Notebook 输出。
    corruption_detected = True  # 标记内容损坏已经被阻止。
    corruption_reason = str(error)  # 保存可审计的 shard 标识。
repaired_state = verify_and_reconstruct(manifest, shard_store)  # 从未损坏冗余副本重新验证和恢复。
repair_error = max(float(np.max(np.abs(repaired_state[name] - checkpoint_state[name]))) for name in checkpoint_state)  # 计算修复状态与原 checkpoint 的最大差。
print(f"错误行为：所有对象存在={existence_only_accepts}，仅存在性检查会接受被改写的{corrupted_shard_id}")  # 展示静默内容损坏风险。
print(f"修正行为：detected={corruption_detected}，reason={corruption_reason}；冗余副本恢复max_error={repair_error:.12f}")  # 展示摘要门禁和修复结果。

错误行为：所有对象存在=True，仅存在性检查会接受被改写的model.weight/rank-0
修正行为：detected=True，reason=checksum_mismatch:model.weight/rank-0；冗余副本恢复max_error=0.000000000000


## 6. 生产边界

内存字典不代表对象存储。生产 checkpoint 需要临时前缀加原子 manifest 提交、跨节点一致性屏障、异步写入回压、Erasure Coding/副本、全局 RNG 与 data-loader 状态、ZeRO/FSDP 平铺元数据、版本兼容和定期故障注入恢复演练。

In [6]:
checkpoint_diagnostics = {"samples": len(samples), "old_world_size": manifest["world_size"], "new_world_size": len(new_world_shards), "global_step": manifest["global_step"], "tensor_count": len(manifest["tensors"]), "baseline_parameter_error": baseline_parameter_error, "baseline_resume_gap": baseline_resume_gap, "manifest_resume_gap": corrected_resume_gap, "checksum_failures": int(corruption_detected)}  # 汇总拓扑变化、恢复精度和完整性指标。
print("生产监控快照：", checkpoint_diagnostics)  # 输出 Elastic 训练恢复应持续监控的信号。

生产监控快照： {'samples': 6, 'old_world_size': 3, 'new_world_size': 2, 'global_step': 3, 'tensor_count': 4, 'baseline_parameter_error': 2.1093739283290374, 'baseline_resume_gap': 0.5144075940748061, 'manifest_resume_gap': 0.0, 'checksum_failures': 1}


## 7. 最小回归测试

断言覆盖真实训练、旧拓扑失败、全局恢复、续训一致性和内容校验。

In [7]:
assert len(samples) >= 5 and all(row["gradient_norm"] > 0.0 for row in pre_checkpoint_history)  # 保证至少五条样本且 checkpoint 前实际执行 backward。
assert baseline_parameter_error > 1.0e-6  # 保证 world size 改变时同编号 rank 恢复真实丢失状态。
assert max(float(np.max(np.abs(corrected_recovered_state[name] - checkpoint_state[name]))) for name in checkpoint_state) < 1.0e-12  # 保证 Manifest 全局坐标完整重建模型和动量。
assert corrected_resume_gap < 1.0e-12 and baseline_resume_gap > corrected_resume_gap + 1.0e-6  # 保证同一下一批次续训与不中断参考一致。
assert corruption_detected and corruption_reason.startswith("checksum_mismatch") and existence_only_accepts  # 保证对象存在但内容损坏的失败真实复现。
assert repair_error < 1.0e-12 and manifest["sampler_cursor"] == checkpoint_metadata["sampler_cursor"]  # 保证冗余修复和数据进度都精确恢复。